# SVG Scaling Laws — Full Training Pipeline

**Runtime**: A100 GPU (Colab Pro)

This notebook runs the entire experiment pipeline:

| # | Section | Est. Time |
|---|---------|----------|
| 1 | Setup (Drive + git pull) | 1 min |
| 2 | Data preprocessing + tokenization | 15 min |
| 3 | SP LR Sweep (Tiny × 5 LRs) | 15 min |
| 4 | SP Scaling Study (5 models) | 2.5 h |
| 5 | µP LR Sweep (Tiny × 7 LRs) | 20 min |
| 6 | µP Scaling Study (5 models) | 2.5 h |
| 7 | Best model extended training | 1.5 h |
| 8 | Generation + Evaluation | 15 min |

**Total: ~7 hours** (set Runtime → A100, then Run All)

Results are saved on Drive so they survive session disconnects.
Each section checks for existing results and skips if already complete.

---
## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/svg-scaling-project
!git pull
!pip install -r requirements.txt -q

In [ ]:
import torch, subprocess, time, json, os, math

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

def run_train(args, label):
    """Run train.py and return summary dict."""
    t0 = time.time()
    result = subprocess.run(['python', 'src/train.py'] + args)
    elapsed = time.time() - t0
    status = 'OK' if result.returncode == 0 else f'FAIL({result.returncode})'
    print(f'  {label}: {status}, {elapsed/60:.1f} min')
    return result.returncode

def load_summary(output_dir):
    """Load summary.json from a run directory."""
    path = os.path.join(output_dir, 'summary.json')
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

def is_done(output_dir):
    """Check if a run already completed."""
    return os.path.exists(os.path.join(output_dir, 'summary.json'))

---
## 2. Data Preprocessing + Tokenization

Downloads `starvector/svg-icons-simple` from HuggingFace, cleans SVGs, trains BPE tokenizer.

If tokens < 100M (due to coordinate normalization shortening SVGs), automatically adds `svg-emoji-simple` as supplementary data.

In [ ]:
import shutil

def tokenize_and_check():
    """Tokenize and return train token count."""
    if os.path.exists('data/tokenized'):
        shutil.rmtree('data/tokenized')
    subprocess.run([
        'python', 'src/tokenize_data.py',
        '--input-dir', 'data/processed',
        '--output-dir', 'data/tokenized',
        '--vocab-size', '4096',
        '--max-token-len', '0',
    ])
    with open('data/tokenized/tokenize_stats.json') as f:
        return json.load(f)['train']['total_tokens']

# Step 1: Preprocess primary dataset
if not os.path.exists('data/processed/train.jsonl'):
    subprocess.run([
        'python', 'src/preprocess.py',
        '--download', 'starvector/svg-icons-simple',
        '--output-dir', 'data/processed',
        '--min-len', '50',
    ])

# Step 2: Tokenize
n_tokens = tokenize_and_check()
print(f"Train tokens: {n_tokens:,}")

# Step 3: If < 100M, add supplementary dataset
if n_tokens < 100_000_000:
    print(f"\n{n_tokens:,} < 100M. Adding svg-emoji-simple...")
    subprocess.run([
        'python', 'src/preprocess.py',
        '--download', 'starvector/svg-emoji-simple',
        '--output-dir', 'data/processed_emoji',
        '--min-len', '50',
    ])
    # Append to main dataset
    for split in ['train', 'val', 'test']:
        src = f'data/processed_emoji/{split}.jsonl'
        dst = f'data/processed/{split}.jsonl'
        if os.path.exists(src):
            with open(src) as f_in, open(dst, 'a') as f_out:
                f_out.write(f_in.read())
            print(f"  Appended {split}.jsonl")

    # Re-tokenize with combined data
    n_tokens = tokenize_and_check()
    print(f"Train tokens after supplement: {n_tokens:,}")

assert n_tokens >= 100_000_000, f'Need >= 100M train tokens, got {n_tokens:,}'
print(f"\n✓ {n_tokens:,} train tokens (>= 100M)")

---
## 3. SP LR Sweep (Tiny × 5 LRs)

In [ ]:
sp_lrs = ['3e-5', '1e-4', '3e-4', '1e-3', '3e-3']
sp_sweep_results = []

print('=== SP LR Sweep ===')
for lr in sp_lrs:
    out = f'results/runs/lr_sweep/tiny_lr_{lr}'
    if is_done(out):
        print(f'  LR={lr}: already done, skipping')
    else:
        run_train(['--config', 'configs/tiny.yaml',
                   '--learning-rate', lr,
                   '--output-dir', out,
                   '--device', 'cuda'], f'SP Tiny LR={lr}')
    s = load_summary(out)
    if s:
        sp_sweep_results.append({'lr': lr, **s})

# Find best
print(f'\n{"LR":>8s}  {"Final Val Loss":>14s}  {"Final PPL":>10s}')
print('-' * 36)
for r in sorted(sp_sweep_results, key=lambda x: x['final_val_loss']):
    print(f'{r["lr"]:>8s}  {r["final_val_loss"]:>14.4f}  {r["final_val_ppl"]:>10.2f}')

sp_best = min(sp_sweep_results, key=lambda x: x['final_val_loss'])
sp_lr = sp_best['lr']
print(f'\n→ SP optimal LR: {sp_lr} (final_val_loss={sp_best["final_val_loss"]:.4f})')

---
## 4. SP Scaling Study (5 Models)

In [ ]:
models = ['tiny', 'small', 'medium', 'large', 'xl']
sp_results = []

print(f'=== SP Scaling Study (LR={sp_lr}) ===')
for name in models:
    out = f'results/runs/sp/{name}'
    if is_done(out):
        print(f'  {name}: already done, skipping')
    else:
        run_train(['--config', f'configs/{name}.yaml',
                   '--learning-rate', sp_lr,
                   '--output-dir', out,
                   '--device', 'cuda'], f'SP {name}')
    s = load_summary(out)
    if s:
        sp_results.append(s)

print(f'\n{"Model":>8s}  {"Params":>8s}  {"Final Val Loss":>14s}  {"Final PPL":>10s}')
print('-' * 46)
for r in sp_results:
    print(f'{r["config_name"]:>8s}  {r["n_params"]/1e6:>7.1f}M  '
          f'{r["final_val_loss"]:>14.4f}  {r["final_val_ppl"]:>10.2f}')

---
## 5. µP LR Sweep (Tiny × 7 LRs)

Higher LRs (1e-2, 3e-2) included — µP should stabilize them.

In [ ]:
mup_lrs = ['3e-5', '1e-4', '3e-4', '1e-3', '3e-3', '1e-2', '3e-2']
mup_sweep_results = []

print('=== µP LR Sweep ===')
for lr in mup_lrs:
    out = f'results/runs/mup_lr_sweep/tiny_lr_{lr}'
    if is_done(out):
        print(f'  LR={lr}: already done, skipping')
    else:
        run_train(['--config', 'configs/tiny.yaml',
                   '--learning-rate', lr,
                   '--output-dir', out,
                   '--device', 'cuda',
                   '--mup'], f'µP Tiny LR={lr}')
    s = load_summary(out)
    if s:
        mup_sweep_results.append({'lr': lr, **s})

print(f'\n{"LR":>8s}  {"Final Val Loss":>14s}  {"Final PPL":>10s}')
print('-' * 36)
for r in sorted(mup_sweep_results, key=lambda x: x['final_val_loss']):
    print(f'{r["lr"]:>8s}  {r["final_val_loss"]:>14.4f}  {r["final_val_ppl"]:>10.2f}')

mup_best = min(mup_sweep_results, key=lambda x: x['final_val_loss'])
mup_lr = mup_best['lr']
print(f'\n→ µP optimal LR: {mup_lr} (final_val_loss={mup_best["final_val_loss"]:.4f})')

---
## 6. µP Scaling Study (5 Models)

In [ ]:
mup_results = []

print(f'=== µP Scaling Study (LR={mup_lr}) ===')
for name in models:
    out = f'results/runs/mup/{name}'
    if is_done(out):
        print(f'  {name}: already done, skipping')
    else:
        run_train(['--config', f'configs/{name}.yaml',
                   '--learning-rate', mup_lr,
                   '--output-dir', out,
                   '--device', 'cuda',
                   '--mup'], f'µP {name}')
    s = load_summary(out)
    if s:
        mup_results.append(s)

print(f'\n{"Model":>8s}  {"Params":>8s}  {"Final Val Loss":>14s}  {"Final PPL":>10s}')
print('-' * 46)
for r in mup_results:
    print(f'{r["config_name"]:>8s}  {r["n_params"]/1e6:>7.1f}M  '
          f'{r["final_val_loss"]:>14.4f}  {r["final_val_ppl"]:>10.2f}')

---
## 7. Best Model Extended Training (Part 4)

Resume µP XL from final checkpoint and train for additional epochs.

In [ ]:
extended_dir = 'results/runs/mup_xl_extended'

if is_done(extended_dir):
    print('Extended training already done, skipping.')
else:
    # Resume from µP XL checkpoint, train for 2x the original steps
    xl_summary = load_summary('results/runs/mup/xl')
    original_steps = xl_summary['total_steps']
    extended_steps = original_steps * 2
    print(f'Original steps: {original_steps}, extending to: {extended_steps}')

    run_train(['--config', 'configs/xl.yaml',
               '--learning-rate', mup_lr,
               '--output-dir', extended_dir,
               '--device', 'cuda',
               '--mup',
               '--resume', 'results/runs/mup/xl/final_checkpoint.pt',
               '--max-steps', str(extended_steps)],
              f'µP XL extended ({extended_steps} steps)')

ext_summary = load_summary(extended_dir)
if ext_summary:
    print(f'\nExtended XL: final_val_loss={ext_summary["final_val_loss"]:.4f}, '
          f'ppl={ext_summary["final_val_ppl"]:.2f}')

---
## 8. Generation + Evaluation

In [ ]:
# Use extended model if available, otherwise µP XL
if os.path.exists(f'{extended_dir}/best_model.pt'):
    ckpt = f'{extended_dir}/best_model.pt'
else:
    ckpt = 'results/runs/mup/xl/best_model.pt'
print(f'Using checkpoint: {ckpt}')

# Unconditional generation at 3 temperatures
for temp in ['0.5', '0.8', '1.0']:
    out = f'results/samples/unconditional/temp_{temp}'
    if os.path.exists(out) and len(os.listdir(out)) > 0:
        print(f'  temp={temp}: samples exist, skipping')
        continue
    print(f'\nGenerating: unconditional, temp={temp}')
    subprocess.run([
        'python', 'src/generate.py',
        '--config', 'configs/xl.yaml',
        '--checkpoint', ckpt,
        '--mup',
        '--num-samples', '10',
        '--temperature', temp,
        '--top-k', '0',
        '--top-p', '0.95',
        '--output-dir', out,
        '--device', 'cuda',
    ])

# Prefix-conditioned generation
prefixes = [
    '<svg viewBox="0 0 24 24"><circle cx="12" cy="12" r="10"',
    '<svg viewBox="0 0 24 24"><path d="M12 2',
    '<svg viewBox="0 0 24 24"><rect x="3" y="3" width="18" height="18"',
]
for i, prefix in enumerate(prefixes):
    out = f'results/samples/prefix/prefix_{i}'
    if os.path.exists(out) and len(os.listdir(out)) > 0:
        print(f'  prefix_{i}: samples exist, skipping')
        continue
    print(f'\nGenerating: prefix_{i}')
    subprocess.run([
        'python', 'src/generate.py',
        '--config', 'configs/xl.yaml',
        '--checkpoint', ckpt,
        '--mup',
        '--num-samples', '5',
        '--prefix', prefix,
        '--temperature', '0.8',
        '--top-k', '0',
        '--top-p', '0.95',
        '--output-dir', out,
        '--device', 'cuda',
    ])

print('\nGeneration complete.')

In [ ]:
# Evaluation
print('=== Evaluation ===')
subprocess.run([
    'python', 'src/evaluate.py',
    '--config', 'configs/xl.yaml',
    '--checkpoint', ckpt,
    '--mup',
    '--samples-dir', 'results/samples/',
    '--test-data', 'data/tokenized/test.bin',
    '--output-dir', 'results/evaluation',
    '--device', 'cuda',
])

# Print results
if os.path.exists('results/evaluation/eval_metrics.json'):
    with open('results/evaluation/eval_metrics.json') as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2))

---
## 9. Summary

All results are saved under `results/` on Drive.

Next steps (on local machine):
1. `git pull` to get results
2. `python scripts/analyze_scaling.py` — SP power law fit
3. `python scripts/analyze_mup.py` — SP vs µP comparison plots
4. Write report

In [ ]:
print('\n' + '=' * 60)
print('PIPELINE COMPLETE')
print('=' * 60)

sections = [
    ('SP LR Sweep',    'results/runs/lr_sweep'),
    ('SP Scaling',     'results/runs/sp'),
    ('µP LR Sweep',   'results/runs/mup_lr_sweep'),
    ('µP Scaling',    'results/runs/mup'),
    ('Extended XL',    extended_dir),
    ('Samples',        'results/samples'),
    ('Evaluation',     'results/evaluation'),
]
for label, path in sections:
    exists = '✓' if os.path.exists(path) else '✗'
    print(f'  {exists} {label:>15s}: {path}')